# Week 9 Companion Notebook: Spark & Distributed Transformation

## ISM 6562 - Big Data for Business Applications

---

This notebook accompanies the Week 9 lecture slides on **Apache Spark and the Transform Layer** of the ELT pipeline. Building on Week 8's HDFS storage layer, you will now learn how Spark reads data from a data lake, applies distributed transformations, and writes curated analytical outputs.

You will work through two complete business scenarios:

- **HealthPulse** — Schema harmonization across hospitals, patient-claims enrichment, device anomaly detection, and lab result analysis
- **GreenRoute** — GPS trip segmentation, delivery enrichment, fuel efficiency analysis, and weather impact correlation

By the end of this notebook, you will have hands-on experience with:

1. Creating and configuring a SparkSession connected to a standalone Spark cluster
2. Reading CSV, JSON, and Parquet data into Spark DataFrames
3. Core transformations: select, filter, groupBy, join, window functions
4. Spark SQL for declarative querying
5. Writing curated Parquet output with partitioning
6. Basic performance optimization: caching, broadcast joins, and query plans

---

## Part 0: Setup & Cluster Verification

The `docker-compose.yml` for this week creates a Spark cluster with the following components:

| Component | Container Name | Role |
|---|---|---|
| **Spark Master** | `spark-master` | Coordinates the cluster, assigns tasks to workers |
| **Spark Worker 1** | `spark-worker-1` | Executes tasks (2 cores, 2 GB RAM) |
| **Spark Worker 2** | `spark-worker-2` | Enables parallelism demos (2 cores, 2 GB RAM) |
| **Jupyter** | `spark-jupyter` | PySpark notebook server (this notebook!) |

**Before running this notebook**, start the cluster from your terminal:

```bash
cd week09-spark-distributed-transformation
docker compose up -d
```

Then open this notebook at: **http://localhost:8888?token=spark**

You can monitor the Spark cluster at:

- **Spark Master UI**: http://localhost:8080
- **Spark Worker 1 UI**: http://localhost:8081
- **Spark Worker 2 UI**: http://localhost:8082
- **Spark Application UI**: http://localhost:4040 (active while a job runs)

In [ ]:
# Create a SparkSession connected to the Spark master
#
# Note: spark.jars.packages pulls the spark-avro connector from Maven on first
# launch. Avro is the right format for high-volume telemetry like our GreenRoute
# GPS data (3-5x smaller than JSON, schema-enforced, native Kafka integration).
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("Week9-CompanionNotebook") \
    .master("spark://spark-master:7077") \
    .config("spark.executor.memory", "1g") \
    .config("spark.driver.memory", "1g") \
    .config("spark.sql.shuffle.partitions", "8") \
    .config("spark.jars.packages", "org.apache.spark:spark-avro_2.12:3.5.0") \
    .getOrCreate()

# Verify the connection
print(f"Spark version: {spark.version}")
print(f"Default parallelism: {spark.sparkContext.defaultParallelism}")
print(f"Application ID: {spark.sparkContext.applicationId}")
print(f"Master: {spark.sparkContext.master}")

In [ ]:
# Download all data files from the course data repository
import os
import subprocess

REPO = "https://raw.githubusercontent.com/prof-tcsmith/ism6562s26-class/main"
DATA_DIR = "/home/jovyan/data"

# Define all files to download
files_to_download = [
    # Week 8 HealthPulse data
    ("week08/data/healthpulse/patient-records/hospital_01_patients.csv", "healthpulse/patient-records/hospital_01_patients.csv"),
    ("week08/data/healthpulse/patient-records/hospital_02_patients.csv", "healthpulse/patient-records/hospital_02_patients.csv"),
    ("week08/data/healthpulse/patient-records/hospital_03_patients.csv", "healthpulse/patient-records/hospital_03_patients.csv"),
    ("week08/data/healthpulse/device-readings/device_readings.json", "healthpulse/device-readings/device_readings.json"),
    ("week08/data/healthpulse/insurance-claims/insurance_claims.csv", "healthpulse/insurance-claims/insurance_claims.csv"),
    # Week 9 HealthPulse data (new)
    ("week09/data/healthpulse/patient-records/hospital_04_patients.csv", "healthpulse/patient-records/hospital_04_patients.csv"),
    ("week09/data/healthpulse/lab-results/lab_results.json", "healthpulse/lab-results/lab_results.json"),
    # Week 8 GreenRoute data
    ("week08/data/greenroute/delivery-receipts/delivery_receipts.json", "greenroute/delivery-receipts/delivery_receipts.json"),
    ("week08/data/greenroute/weather/weather_data.json", "greenroute/weather/weather_data.json"),
    # GPS schema file (the .avsc lives at the gps-telemetry root)
    ("week08/data/greenroute/gps-telemetry/gps_reading.avsc", "greenroute/gps-telemetry/gps_reading.avsc"),
    # Week 9 GreenRoute data (new)
    ("week09/data/greenroute/route-plans/route_plans.json", "greenroute/route-plans/route_plans.json"),
    ("week09/data/greenroute/fuel-logs/fuel_logs.csv", "greenroute/fuel-logs/fuel_logs.csv"),
]

# Download GPS telemetry partitions (days 01-03, all 24 hours each)
# GPS is Avro -- the right format for high-volume binary telemetry
for day in range(1, 4):
    for hour in range(24):
        remote = f"week08/data/greenroute/gps-telemetry/day_{day:02d}/hour_{hour:02d}/gps_data.avro"
        local = f"greenroute/gps-telemetry/day_{day:02d}/hour_{hour:02d}/gps_data.avro"
        files_to_download.append((remote, local))

# Create directories and download
downloaded = 0
skipped = 0
for remote_path, local_path in files_to_download:
    full_local = os.path.join(DATA_DIR, local_path)
    os.makedirs(os.path.dirname(full_local), exist_ok=True)
    if os.path.exists(full_local):
        skipped += 1
        continue
    result = subprocess.run(
        ["curl", "-sL", f"{REPO}/{remote_path}", "-o", full_local],
        capture_output=True, text=True
    )
    if result.returncode == 0:
        downloaded += 1
    else:
        print(f"  WARNING: Failed to download {remote_path}")

print(f"Download complete: {downloaded} new files, {skipped} already present.")
print(f"Total files: {downloaded + skipped}")

In [ ]:
# Verify the data directory structure
import os

for root, dirs, files in os.walk(DATA_DIR):
    # Skip deep GPS subdirectories — just show the day level
    level = root.replace(DATA_DIR, "").count(os.sep)
    indent = "  " * level
    if level <= 3:
        print(f"{indent}{os.path.basename(root)}/")
    if level <= 2:
        for f in sorted(files):
            print(f"{indent}  {f}")

---

## Part 1: Spark Fundamentals Review

Before diving into the business scenarios, let's review the core Spark DataFrame operations. This section maps directly to the lecture slides on Spark transformations, actions, and the Catalyst optimizer.

### 1a. Reading Data into Spark

Spark supports multiple file formats out of the box. Let's compare reading CSV, JSON, and see how the schema inference works for each.

In [ ]:
# Read a CSV file with header and schema inference
patients_h01 = spark.read.csv(
    "/home/jovyan/data/healthpulse/patient-records/hospital_01_patients.csv",
    header=True,
    inferSchema=True
)

print("=== Hospital 01 Patients (CSV) ===")
print(f"Rows: {patients_h01.count()}, Columns: {len(patients_h01.columns)}")
print("\nSchema:")
patients_h01.printSchema()
print("\nFirst 5 rows:")
patients_h01.show(5, truncate=False)

In [ ]:
# Read a JSON file — note how Spark handles nested structures
device_readings = spark.read.json(
    "/home/jovyan/data/healthpulse/device-readings/device_readings.json"
)

print("=== Device Readings (JSON) ===")
print(f"Rows: {device_readings.count()}, Columns: {len(device_readings.columns)}")
print("\nSchema (notice the nested 'reading' struct):")
device_readings.printSchema()
print("\nFirst 3 rows:")
device_readings.show(3, truncate=False)

**Key Takeaway:** CSV requires `header=True` and `inferSchema=True` to get proper column names and types. JSON automatically infers the schema, including nested structures. Later in this notebook, we will also read and write **Parquet**, which stores the schema in the file metadata — no inference needed.

### 1b. Core Transformations

Transformations are **lazy** — they define a computation but do not execute it until an action is called. The key transformations are `select`, `filter`, `withColumn`, `groupBy`, and `orderBy`.

In [ ]:
from pyspark.sql import functions as F

# select: choose specific columns
patients_slim = patients_h01.select("patient_id", "name", "age", "dept")
patients_slim.show(5)

# filter: keep only rows matching a condition
elderly_patients = patients_h01.filter(F.col("age") >= 65)
print(f"Patients age >= 65: {elderly_patients.count()}")

# withColumn: add or transform a column
patients_with_age_group = patients_h01.withColumn(
    "age_group",
    F.when(F.col("age") < 18, "Pediatric")
     .when(F.col("age") < 65, "Adult")
     .otherwise("Senior")
)
patients_with_age_group.select("patient_id", "age", "age_group").show(10)

In [ ]:
# groupBy + agg: aggregate data
dept_stats = patients_h01.groupBy("dept").agg(
    F.count("*").alias("patient_count"),
    F.round(F.avg("age"), 1).alias("avg_age"),
    F.min("age").alias("min_age"),
    F.max("age").alias("max_age")
)

# orderBy: sort results
dept_stats.orderBy(F.desc("patient_count")).show()

### 1c. Spark SQL

Spark allows you to register DataFrames as temporary views and query them with standard SQL. Under the hood, both the DataFrame API and Spark SQL use the same Catalyst optimizer — they produce identical execution plans.

In [ ]:
# Register the DataFrame as a temporary SQL view
patients_h01.createOrReplaceTempView("patients")

# Run a SQL query
result_sql = spark.sql("""
    SELECT dept,
           COUNT(*) AS patient_count,
           ROUND(AVG(age), 1) AS avg_age
    FROM patients
    GROUP BY dept
    ORDER BY patient_count DESC
""")

print("=== SQL Query Result ===")
result_sql.show()

# Compare: DataFrame API produces the same result
result_df = patients_h01.groupBy("dept").agg(
    F.count("*").alias("patient_count"),
    F.round(F.avg("age"), 1).alias("avg_age")
).orderBy(F.desc("patient_count"))

print("=== DataFrame API Result (same query) ===")
result_df.show()

### 1d. Lazy Evaluation: Transformations vs Actions

Spark transformations are **lazy** — nothing executes until you call an **action** (like `.show()`, `.count()`, or `.collect()`). This allows Spark's Catalyst optimizer to reorder and optimize the entire computation graph before execution.

Use `.explain()` to see the physical plan that Spark will execute.

In [ ]:
# Build a chain of transformations — nothing executes yet!
pipeline = (
    patients_h01
    .filter(F.col("age") >= 30)
    .withColumn("age_decade", (F.floor(F.col("age") / 10) * 10).cast("int"))
    .groupBy("dept", "age_decade")
    .agg(F.count("*").alias("count"))
    .orderBy("dept", "age_decade")
)

# This does NOT execute the query — it only shows the plan
print("=== Physical Execution Plan ===")
pipeline.explain()
print("\n(No data has been processed yet — the plan above is just a blueprint.)")

In [ ]:
%%time
# NOW trigger execution with an action
print(f"Result row count: {pipeline.count()}")
pipeline.show(10)

### 1e. Joins

Joins are one of the most important operations in data engineering. Spark supports inner, left, right, full outer, cross, and semi joins. When one side of a join is small enough to fit in memory, Spark can use a **broadcast join** to avoid shuffling the larger dataset.

In [ ]:
# Read insurance claims
claims = spark.read.csv(
    "/home/jovyan/data/healthpulse/insurance-claims/insurance_claims.csv",
    header=True,
    inferSchema=True
)
print(f"Claims: {claims.count()} rows")
claims.printSchema()

# Join patients with claims on patient_id
enriched = patients_h01.join(claims, on="patient_id", how="inner")
print(f"\nEnriched (patients + claims): {enriched.count()} rows")
enriched.show(5, truncate=False)

In [ ]:
# Broadcast join: when one table is small, broadcast it to all workers
# This avoids a full shuffle of the large table across the network

# Create a small lookup table for department descriptions
dept_lookup = spark.createDataFrame([
    ("Cardiology", "Heart and cardiovascular system"),
    ("Neurology", "Brain and nervous system"),
    ("Oncology", "Cancer treatment"),
    ("Pediatrics", "Children's health"),
    ("Orthopedics", "Bones and joints"),
    ("Emergency", "Emergency medicine"),
], ["dept", "description"])

# Broadcast the small table
enriched_with_desc = patients_h01.join(
    F.broadcast(dept_lookup),
    on="dept",
    how="left"
)

# Check the plan — you should see "BroadcastHashJoin"
print("=== Join Plan (look for BroadcastHashJoin) ===")
enriched_with_desc.explain()

enriched_with_desc.select("patient_id", "dept", "description").show(5)

---

## Part 2: Business Scenario — HealthPulse Transforms

HealthPulse aggregates data from four hospitals, each with slightly different schemas. In this section, you will:

1. **Harmonize schemas** across all four hospital CSV files
2. **Enrich patients with insurance claims** to compute billing analytics
3. **Analyze device readings** to flag health anomalies
4. **Process lab results** to identify critical findings per hospital
5. **Write curated Parquet** output for downstream analytics

### Step 2.1: Load All HealthPulse Data

We already downloaded all files in Part 0. Let's read each hospital CSV and inspect their schemas to understand the harmonization challenge.

In [ ]:
# Read all four hospital CSV files
h01 = spark.read.csv("/home/jovyan/data/healthpulse/patient-records/hospital_01_patients.csv", header=True, inferSchema=True)
h02 = spark.read.csv("/home/jovyan/data/healthpulse/patient-records/hospital_02_patients.csv", header=True, inferSchema=True)
h03 = spark.read.csv("/home/jovyan/data/healthpulse/patient-records/hospital_03_patients.csv", header=True, inferSchema=True)
h04 = spark.read.csv("/home/jovyan/data/healthpulse/patient-records/hospital_04_patients.csv", header=True, inferSchema=True)

# Compare their schemas side by side
for name, df in [("Hospital 01", h01), ("Hospital 02", h02), ("Hospital 03", h03), ("Hospital 04", h04)]:
    print(f"\n=== {name} ===")
    print(f"  Rows: {df.count()}, Columns: {len(df.columns)}")
    print(f"  Column names: {df.columns}")

Notice the problem: the department column has a **different name** in each hospital file:

| Hospital | Department Column Name | Extra Columns |
|----------|----------------------|---------------|
| Hospital 01 | `dept` | — |
| Hospital 02 | `department` | — |
| Hospital 03 | `dept` or variant | May have extra columns |
| Hospital 04 | `department_name` | `emergency_contact` |

This is a **very common** real-world challenge. When multiple source systems feed into a data lake, schema differences must be resolved in the Transform layer before analytics can run across the combined dataset.

### Step 2.2: Schema Harmonization

We will standardize all four hospital DataFrames to a common schema with these columns:

- `patient_id`, `name`, `age`, `gender`, `department`, `hospital_id`

Any extra columns will be dropped, and the department column will be renamed to `department` in all cases.

In [ ]:
def harmonize_hospital(df, hospital_id, dept_col_name):
    """
    Standardize a hospital DataFrame to the common schema.
    
    Parameters:
        df: Input DataFrame
        hospital_id: String identifier for the hospital
        dept_col_name: The name of the department column in this file
    
    Returns:
        DataFrame with standardized columns
    """
    return df.select(
        F.col("patient_id"),
        F.col("name"),
        F.col("age"),
        F.col("gender"),
        F.col(dept_col_name).alias("department")
    ).withColumn("hospital_id", F.lit(hospital_id))


# Identify the department column name in each file
print("Identifying department column in each file:")
for name, df in [("H01", h01), ("H02", h02), ("H03", h03), ("H04", h04)]:
    dept_candidates = [c for c in df.columns if "dept" in c.lower() or "department" in c.lower()]
    print(f"  {name}: {dept_candidates}")

In [ ]:
# Harmonize each hospital — adjust dept_col_name based on the output above
# (Update these if the actual column names differ from what is shown)
h01_std = harmonize_hospital(h01, "hospital_01", "dept")
h02_std = harmonize_hospital(h02, "hospital_02", "department")

# Hospital 03 — check which column name it uses
h03_dept_col = [c for c in h03.columns if "dept" in c.lower() or "department" in c.lower()][0]
h03_std = harmonize_hospital(h03, "hospital_03", h03_dept_col)

# Hospital 04 uses "department_name"
h04_dept_col = [c for c in h04.columns if "dept" in c.lower() or "department" in c.lower()][0]
h04_std = harmonize_hospital(h04, "hospital_04", h04_dept_col)

# Union all into a single DataFrame
all_patients = h01_std.union(h02_std).union(h03_std).union(h04_std)

print(f"Unified patients: {all_patients.count()} rows")
print(f"Schema:")
all_patients.printSchema()

# Verify counts per hospital
all_patients.groupBy("hospital_id").count().orderBy("hospital_id").show()

### Step 2.3: Patient-Claims Enrichment

Now that we have a unified patient dataset, let's join it with insurance claims to compute billing analytics per patient.

In [ ]:
# Read insurance claims
claims = spark.read.csv(
    "/home/jovyan/data/healthpulse/insurance-claims/insurance_claims.csv",
    header=True,
    inferSchema=True
)

print(f"Insurance claims: {claims.count()} rows")
claims.printSchema()
claims.show(3, truncate=False)

In [ ]:
# Join patients with claims
patient_claims = all_patients.join(claims, on="patient_id", how="inner")

print(f"Patient-claims joined: {patient_claims.count()} rows")

# Compute billing analytics per patient
patient_billing = patient_claims.groupBy(
    "patient_id", "name", "hospital_id", "department"
).agg(
    F.count("*").alias("claim_count"),
    F.round(F.sum("amount"), 2).alias("total_billed"),
    F.round(F.avg("amount"), 2).alias("avg_claim_amount"),
    F.round(
        F.sum(F.when(F.col("status") == "approved", 1).otherwise(0)) / F.count("*") * 100, 1
    ).alias("approval_rate_pct")
)

# Top 10 patients by total billed
print("\n=== Top 10 Patients by Total Billed ===")
patient_billing.orderBy(F.desc("total_billed")).show(10, truncate=False)

### Step 2.4: Device Reading Analytics

Device readings contain nested JSON structures with vital sign measurements. We will parse these, compute per-patient averages, and flag anomalies based on clinical thresholds:

| Vital Sign | Anomaly Threshold |
|-----------|-------------------|
| Heart Rate | > 100 BPM |
| SpO2 | < 92% |
| Blood Pressure (systolic) | > 160 mmHg |

In [ ]:
# Re-read device readings
device_readings = spark.read.json(
    "/home/jovyan/data/healthpulse/device-readings/device_readings.json"
)

print(f"Device readings: {device_readings.count()} rows")
device_readings.printSchema()
device_readings.show(3, truncate=False)

In [ ]:
# Flatten the nested reading struct
# First, inspect what fields are inside the reading struct
print("Columns available:")
for col_name in device_readings.columns:
    print(f"  {col_name}: {device_readings.schema[col_name].dataType}")

# Flatten — select nested fields using dot notation
# Adjust field names based on the actual schema above
readings_flat = device_readings.select(
    "patient_id",
    "device_type",
    "timestamp",
    "hospital_id",
    F.col("reading.*")  # Expands all fields in the reading struct
)

print("\nFlattened schema:")
readings_flat.printSchema()
readings_flat.show(5, truncate=False)

In [ ]:
# Compute per-patient averages by device type
patient_device_avgs = readings_flat.groupBy("patient_id", "device_type").agg(
    F.count("*").alias("reading_count"),
    *[
        F.round(F.avg(c), 2).alias(f"avg_{c}")
        for c in readings_flat.columns
        if c not in ("patient_id", "device_type", "timestamp", "hospital_id")
        and readings_flat.schema[c].dataType.typeName() in ("double", "integer", "long", "float")
    ]
)

print("Per-patient averages by device type:")
patient_device_avgs.show(10, truncate=False)

In [ ]:
# Flag anomalies based on clinical thresholds
# We need to check available columns and apply the appropriate thresholds
print("Available numeric columns for anomaly detection:")
numeric_cols = [
    c for c in readings_flat.columns
    if c not in ("patient_id", "device_type", "timestamp", "hospital_id")
    and readings_flat.schema[c].dataType.typeName() in ("double", "integer", "long", "float")
]
print(f"  {numeric_cols}")

# Build anomaly conditions dynamically based on available columns
anomaly_conditions = []

# Heart rate anomaly
hr_cols = [c for c in numeric_cols if "heart" in c.lower() or "hr" == c.lower() or "bpm" in c.lower() or "pulse" in c.lower()]
if hr_cols:
    anomaly_conditions.append((hr_cols[0], F.col(hr_cols[0]) > 100, "high_heart_rate"))
    
# SpO2 anomaly
spo2_cols = [c for c in numeric_cols if "spo2" in c.lower() or "oxygen" in c.lower() or "o2" in c.lower()]
if spo2_cols:
    anomaly_conditions.append((spo2_cols[0], F.col(spo2_cols[0]) < 92, "low_spo2"))

# Blood pressure systolic anomaly
bp_cols = [c for c in numeric_cols if "systolic" in c.lower() or "bp_sys" in c.lower() or "sbp" in c.lower()]
if bp_cols:
    anomaly_conditions.append((bp_cols[0], F.col(bp_cols[0]) > 160, "high_bp_systolic"))

print(f"\nAnomaly rules defined: {len(anomaly_conditions)}")
for src_col, _, label in anomaly_conditions:
    print(f"  {label} -> based on column '{src_col}'")

# Apply anomaly flags
flagged = readings_flat
for src_col, condition, label in anomaly_conditions:
    flagged = flagged.withColumn(label, condition.cast("int"))

# Count anomalies per hospital
if anomaly_conditions:
    anomaly_labels = [label for _, _, label in anomaly_conditions]
    anomaly_counts = flagged.groupBy("hospital_id").agg(
        F.count("*").alias("total_readings"),
        *[F.sum(label).alias(f"{label}_count") for label in anomaly_labels]
    )
    print("\n=== Anomaly Counts per Hospital ===")
    anomaly_counts.orderBy("hospital_id").show(truncate=False)
else:
    print("\nNo matching anomaly columns found — inspect the schema above and adjust thresholds.")

### Step 2.5: Lab Result Analysis

The new Week 9 dataset includes 30,000 lab results. We will identify critical results and aggregate them by hospital and test type.

In [ ]:
# Read lab results
lab_results = spark.read.json(
    "/home/jovyan/data/healthpulse/lab-results/lab_results.json"
)

print(f"Lab results: {lab_results.count()} rows")
lab_results.printSchema()
lab_results.show(5, truncate=False)

In [ ]:
# Filter critical results
# Look for a flag/status column that indicates critical results
print("Lab results columns:", lab_results.columns)

# Identify the flag column (could be 'flag', 'status', 'result_flag', etc.)
flag_cols = [c for c in lab_results.columns if "flag" in c.lower() or "status" in c.lower() or "critical" in c.lower()]
print(f"Potential flag columns: {flag_cols}")

if flag_cols:
    flag_col = flag_cols[0]
    print(f"\nUsing '{flag_col}' column. Distinct values:")
    lab_results.groupBy(flag_col).count().orderBy(F.desc("count")).show()
    
    # Filter for critical results
    critical_labs = lab_results.filter(
        F.lower(F.col(flag_col)).contains("critical")
    )
    print(f"Critical lab results: {critical_labs.count()} rows")
else:
    print("No flag column found — using all results for analysis.")
    critical_labs = lab_results

In [ ]:
# Join critical labs with unified patients to get hospital context
critical_with_hospital = critical_labs.join(
    all_patients.select("patient_id", "hospital_id", "department"),
    on="patient_id",
    how="inner"
)

# Identify the test type column
test_cols = [c for c in lab_results.columns if "test" in c.lower() or "type" in c.lower() or "name" in c.lower()]
print(f"Potential test type columns: {test_cols}")

if test_cols:
    test_col = test_cols[0]
    # Aggregate: critical results per hospital per test type
    critical_summary = critical_with_hospital.groupBy(
        "hospital_id", test_col
    ).agg(
        F.count("*").alias("critical_count")
    ).orderBy("hospital_id", F.desc("critical_count"))
    
    print(f"\n=== Critical Lab Results per Hospital per Test Type ===")
    critical_summary.show(20, truncate=False)
else:
    # Fallback: just count by hospital
    critical_summary = critical_with_hospital.groupBy("hospital_id").agg(
        F.count("*").alias("critical_count")
    ).orderBy("hospital_id")
    print("\n=== Critical Lab Results per Hospital ===")
    critical_summary.show()

### Step 2.6: Write Curated Parquet Output

The final step in any Transform pipeline is writing the curated data for downstream consumption. Parquet is the standard output format because it provides:

- **Columnar storage** — only the columns you query are read from disk
- **Built-in compression** — typically 3-10x smaller than CSV
- **Embedded schema** — no need for `inferSchema`, no parsing errors
- **Predicate pushdown** — filters can skip entire row groups

In [ ]:
OUTPUT_DIR = "/home/jovyan/data/healthpulse/curated"

# Write unified patients partitioned by hospital_id
all_patients.write.mode("overwrite") \
    .partitionBy("hospital_id") \
    .parquet(f"{OUTPUT_DIR}/unified_patients")

print("Wrote unified_patients as Parquet (partitioned by hospital_id)")

# Write patient billing summary
patient_billing.write.mode("overwrite") \
    .parquet(f"{OUTPUT_DIR}/patient_billing_summary")

print("Wrote patient_billing_summary as Parquet")

# Show the directory structure created
import os

print("\n=== Curated Output Directory Structure ===")
for root, dirs, files in os.walk(OUTPUT_DIR):
    level = root.replace(OUTPUT_DIR, "").count(os.sep)
    indent = "  " * level
    print(f"{indent}{os.path.basename(root)}/")
    for f in sorted(files)[:5]:  # Show up to 5 files per directory
        print(f"{indent}  {f}")
    if len(files) > 5:
        print(f"{indent}  ... and {len(files) - 5} more files")

In [ ]:
# Compare file sizes: CSV input vs Parquet output
import os

def get_dir_size(path):
    """Get total size of all files in a directory tree."""
    total = 0
    for root, dirs, files in os.walk(path):
        for f in files:
            total += os.path.getsize(os.path.join(root, f))
    return total

csv_size = sum(
    os.path.getsize(f"/home/jovyan/data/healthpulse/patient-records/hospital_{i:02d}_patients.csv")
    for i in range(1, 5)
    if os.path.exists(f"/home/jovyan/data/healthpulse/patient-records/hospital_{i:02d}_patients.csv")
)
parquet_size = get_dir_size(f"{OUTPUT_DIR}/unified_patients")

print(f"=== HealthPulse Size Comparison ===")
print(f"  CSV input (4 hospital files): {csv_size / 1024 / 1024:.2f} MB")
print(f"  Parquet output (partitioned):  {parquet_size / 1024 / 1024:.2f} MB")
if csv_size > 0:
    print(f"  Compression ratio:             {csv_size / parquet_size:.1f}x")

---

## Part 3: Business Scenario — GreenRoute Transforms

GreenRoute is a logistics company that tracks its fleet of delivery trucks via GPS telemetry. In this section, you will:

1. **Read partitioned GPS data** spanning multiple days
2. **Segment trips** using window functions to detect stops
3. **Enrich deliveries** with GPS location context
4. **Analyze fuel efficiency** across the fleet
5. **Correlate weather conditions** with delivery performance
6. **Write curated analytics** as Parquet

### Step 3.1: Verify GreenRoute Data

We downloaded 3 days of GPS telemetry (72 hourly partitions), plus delivery receipts, weather data, route plans, and fuel logs in Part 0. Let's verify the data is in place.

In [ ]:
# Count GPS partition files
import glob

gps_files = glob.glob("/home/jovyan/data/greenroute/gps-telemetry/day_*/hour_*/gps_data.avro")
print(f"GPS partition files found: {len(gps_files)}")

# Check other data files
other_files = [
    ("/home/jovyan/data/greenroute/delivery-receipts/delivery_receipts.json", "Delivery receipts"),
    ("/home/jovyan/data/greenroute/weather/weather_data.json", "Weather data"),
    ("/home/jovyan/data/greenroute/route-plans/route_plans.json", "Route plans"),
    ("/home/jovyan/data/greenroute/fuel-logs/fuel_logs.csv", "Fuel logs"),
]

for path, label in other_files:
    exists = os.path.exists(path)
    size = os.path.getsize(path) / 1024 if exists else 0
    status = f"{size:.1f} KB" if exists else "MISSING"
    print(f"  {label}: {status}")

### Step 3.2: Reading Partitioned GPS Data

One of Spark's strengths is reading data from many files in a single call. We will use a wildcard path to read all GPS partitions at once.

In [ ]:
# Read all GPS partition files at once using a wildcard path
# Avro is the right format here -- 3-5x smaller than JSON, schema-enforced,
# and the schema is embedded in each file so Spark infers it for free.
gps_data = spark.read.format("avro").load(
    "/home/jovyan/data/greenroute/gps-telemetry/day_*/hour_*/gps_data.avro"
)

print(f"GPS records loaded: {gps_data.count()}")
print(f"Partitions: {gps_data.rdd.getNumPartitions()}")
print("\nSchema (read directly from the Avro file headers):")
gps_data.printSchema()
print("\nSample records:")
gps_data.show(5, truncate=False)

### Step 3.3: Trip Segmentation with Window Functions

Window functions let you compute values across a set of rows that are related to the current row — without collapsing them into a single output row (unlike `groupBy`). This is essential for time-series analysis.

We will use window functions to:

1. Order GPS readings per truck by timestamp
2. Identify stops (speed < 2 mph)
3. Assign trip segment IDs (a new segment starts after each stop)
4. Compute duration and distance per trip segment

In [ ]:
from pyspark.sql.window import Window

# First, inspect the GPS data columns to find speed and truck identifiers
print("GPS columns:", gps_data.columns)
gps_data.show(3, truncate=False)

# Identify key columns
truck_cols = [c for c in gps_data.columns if "truck" in c.lower() or "vehicle" in c.lower() or "id" in c.lower()]
speed_cols = [c for c in gps_data.columns if "speed" in c.lower()]
time_cols = [c for c in gps_data.columns if "time" in c.lower() or "ts" in c.lower()]

print(f"\nTruck ID candidates: {truck_cols}")
print(f"Speed candidates: {speed_cols}")
print(f"Timestamp candidates: {time_cols}")

In [ ]:
# Define the truck and speed columns (adjust based on output above)
# Common patterns: truck_id, speed, timestamp
TRUCK_COL = [c for c in gps_data.columns if "truck" in c.lower()][0] if [c for c in gps_data.columns if "truck" in c.lower()] else "truck_id"
SPEED_COL = [c for c in gps_data.columns if "speed" in c.lower()][0] if [c for c in gps_data.columns if "speed" in c.lower()] else "speed"
TIME_COL = [c for c in gps_data.columns if "timestamp" in c.lower()][0] if [c for c in gps_data.columns if "timestamp" in c.lower()] else "timestamp"

print(f"Using columns: truck={TRUCK_COL}, speed={SPEED_COL}, time={TIME_COL}")

# Define a window partitioned by truck, ordered by timestamp
truck_window = Window.partitionBy(TRUCK_COL).orderBy(TIME_COL)

# Step 1: Flag stops (speed < 2)
gps_with_stops = gps_data.withColumn(
    "is_stop",
    F.when(F.col(SPEED_COL) < 2, 1).otherwise(0)
)

# Step 2: Create a running sum of stops — each time a stop occurs, the segment ID increments
gps_with_segments = gps_with_stops.withColumn(
    "trip_segment",
    F.sum("is_stop").over(truck_window)
)

print(f"\nSample GPS data with trip segments:")
gps_with_segments.select(TRUCK_COL, TIME_COL, SPEED_COL, "is_stop", "trip_segment") \
    .orderBy(TRUCK_COL, TIME_COL) \
    .show(15, truncate=False)

In [ ]:
# Step 3: Compute trip-level metrics
trip_metrics = gps_with_segments.filter(F.col("is_stop") == 0).groupBy(
    TRUCK_COL, "trip_segment"
).agg(
    F.min(TIME_COL).alias("trip_start"),
    F.max(TIME_COL).alias("trip_end"),
    F.count("*").alias("gps_points"),
    F.round(F.avg(SPEED_COL), 1).alias("avg_speed"),
    F.round(F.max(SPEED_COL), 1).alias("max_speed")
)

# Compute trip duration in minutes
trip_metrics = trip_metrics.withColumn(
    "duration_minutes",
    F.round(
        (F.unix_timestamp("trip_end") - F.unix_timestamp("trip_start")) / 60, 1
    )
)

print(f"Trip segments identified: {trip_metrics.count()}")
print("\n=== Sample Trip Metrics ===")
trip_metrics.orderBy(TRUCK_COL, "trip_start").show(15, truncate=False)

### Step 3.4: Delivery Enrichment

Delivery receipts contain nested JSON with item details. We will explode the nested arrays, then join with GPS data to add location context to each delivery.

In [ ]:
# Read delivery receipts
deliveries = spark.read.json(
    "/home/jovyan/data/greenroute/delivery-receipts/delivery_receipts.json"
)

print(f"Delivery receipts: {deliveries.count()} rows")
deliveries.printSchema()
deliveries.show(3, truncate=False)

In [ ]:
# Identify array columns to explode
from pyspark.sql.types import ArrayType

array_cols = [
    c for c in deliveries.columns
    if isinstance(deliveries.schema[c].dataType, ArrayType)
]
print(f"Array columns to explode: {array_cols}")

# Explode the items array (if present)
if array_cols:
    items_col = array_cols[0]
    deliveries_exploded = deliveries.withColumn("item", F.explode(F.col(items_col)))
    
    # Select useful fields from the exploded item struct
    print(f"\nExploded '{items_col}' — new schema:")
    deliveries_exploded.printSchema()
    deliveries_exploded.show(5, truncate=False)
else:
    print("No array columns found — using deliveries as-is.")
    deliveries_exploded = deliveries

In [ ]:
# Compute delivery metrics
# Identify the truck ID column in deliveries
delivery_truck_col = [c for c in deliveries.columns if "truck" in c.lower()]
delivery_time_col = [c for c in deliveries.columns if "time" in c.lower() or "date" in c.lower()]

print(f"Delivery truck column: {delivery_truck_col}")
print(f"Delivery time column: {delivery_time_col}")

if delivery_truck_col and delivery_time_col:
    dtruck = delivery_truck_col[0]
    dtime = delivery_time_col[0]
    
    # Delivery metrics: count and timing per truck
    delivery_metrics = deliveries.groupBy(dtruck).agg(
        F.count("*").alias("delivery_count"),
        F.min(dtime).alias("first_delivery"),
        F.max(dtime).alias("last_delivery")
    )
    
    print("\n=== Delivery Metrics per Truck ===")
    delivery_metrics.orderBy(F.desc("delivery_count")).show(10, truncate=False)
else:
    print("Could not identify truck/time columns — inspect schema above.")

### Step 3.5: Fuel Efficiency Analysis

Fuel logs track refueling events. By joining with GPS odometer data, we can compute miles-per-gallon for each truck and identify the least and most fuel-efficient vehicles in the fleet.

In [ ]:
# Read fuel logs
fuel_logs = spark.read.csv(
    "/home/jovyan/data/greenroute/fuel-logs/fuel_logs.csv",
    header=True,
    inferSchema=True
)

print(f"Fuel logs: {fuel_logs.count()} rows")
fuel_logs.printSchema()
fuel_logs.show(5, truncate=False)

In [ ]:
# Identify columns for fuel analysis
fuel_truck_col = [c for c in fuel_logs.columns if "truck" in c.lower() or "vehicle" in c.lower()]
fuel_gallons_col = [c for c in fuel_logs.columns if "gallon" in c.lower() or "fuel" in c.lower() or "litre" in c.lower() or "liter" in c.lower() or "amount" in c.lower()]
fuel_odometer_col = [c for c in fuel_logs.columns if "odometer" in c.lower() or "mile" in c.lower() or "km" in c.lower() or "distance" in c.lower()]
fuel_time_col = [c for c in fuel_logs.columns if "time" in c.lower() or "date" in c.lower()]

print(f"Truck column: {fuel_truck_col}")
print(f"Fuel amount column: {fuel_gallons_col}")
print(f"Odometer column: {fuel_odometer_col}")
print(f"Time column: {fuel_time_col}")

In [ ]:
# Compute fuel efficiency per truck using window functions
# We calculate miles between fill-ups divided by gallons

if fuel_truck_col and fuel_gallons_col:
    ftruck = fuel_truck_col[0]
    fgallons = fuel_gallons_col[0]
    ftime = fuel_time_col[0] if fuel_time_col else None
    fodometer = fuel_odometer_col[0] if fuel_odometer_col else None
    
    if fodometer and ftime:
        # Window to get previous odometer reading
        fuel_window = Window.partitionBy(ftruck).orderBy(ftime)
        
        fuel_with_prev = fuel_logs.withColumn(
            "prev_odometer",
            F.lag(fodometer).over(fuel_window)
        ).withColumn(
            "miles_driven",
            F.col(fodometer) - F.col("prev_odometer")
        ).withColumn(
            "mpg",
            F.round(F.col("miles_driven") / F.col(fgallons), 2)
        ).filter(F.col("prev_odometer").isNotNull())  # Skip first fill-up per truck
        
        # Aggregate MPG per truck
        truck_efficiency = fuel_with_prev.groupBy(ftruck).agg(
            F.count("*").alias("fill_ups"),
            F.round(F.sum("miles_driven"), 1).alias("total_miles"),
            F.round(F.sum(fgallons), 1).alias("total_gallons"),
            F.round(F.avg("mpg"), 2).alias("avg_mpg")
        ).orderBy(F.desc("avg_mpg"))
        
        print("=== Fuel Efficiency by Truck (best to worst) ===")
        truck_efficiency.show(20, truncate=False)
        
        print("\n=== Top 3 Most Efficient Trucks ===")
        truck_efficiency.orderBy(F.desc("avg_mpg")).show(3, truncate=False)
        
        print("\n=== Top 3 Least Efficient Trucks ===")
        truck_efficiency.orderBy("avg_mpg").show(3, truncate=False)
    else:
        # Simple aggregation if no odometer data
        print("No odometer column found — computing total fuel consumed per truck.")
        fuel_summary = fuel_logs.groupBy(ftruck).agg(
            F.count("*").alias("fill_ups"),
            F.round(F.sum(fgallons), 1).alias("total_fuel")
        ).orderBy(F.desc("total_fuel"))
        fuel_summary.show(20, truncate=False)
else:
    print("Could not identify truck/fuel columns — inspect schema above.")

### Step 3.6: Weather Impact Analysis

The weather dataset is relatively small (3,360 records), making it an ideal candidate for a **broadcast join**. We will join weather conditions with delivery data to determine whether adverse weather affects delivery times.

In [ ]:
# Read weather data
weather = spark.read.json(
    "/home/jovyan/data/greenroute/weather/weather_data.json"
)

print(f"Weather records: {weather.count()} rows")
weather.printSchema()
weather.show(5, truncate=False)

In [ ]:
# Identify weather condition and time columns
condition_cols = [c for c in weather.columns if "condition" in c.lower() or "weather" in c.lower() or "type" in c.lower()]
weather_time_cols = [c for c in weather.columns if "time" in c.lower() or "date" in c.lower() or "hour" in c.lower()]

print(f"Condition columns: {condition_cols}")
print(f"Time columns: {weather_time_cols}")

if condition_cols:
    cond_col = condition_cols[0]
    print(f"\nWeather condition distribution:")
    weather.groupBy(cond_col).count().orderBy(F.desc("count")).show()

In [ ]:
# Broadcast join weather with deliveries
# We need to find a common join key (typically date/hour or location)

if condition_cols and delivery_time_col and weather_time_cols:
    cond_col = condition_cols[0]
    
    # Extract date/hour from both datasets for joining
    # Adjust the timestamp parsing based on actual column formats
    weather_prepped = weather.withColumn(
        "join_date", F.to_date(F.col(weather_time_cols[0]))
    )
    
    deliveries_prepped = deliveries.withColumn(
        "join_date", F.to_date(F.col(delivery_time_col[0]))
    )
    
    # Average weather conditions per day
    daily_weather = weather_prepped.groupBy("join_date", cond_col).agg(
        F.count("*").alias("weather_readings")
    )
    
    # Get the dominant weather condition per day
    weather_window = Window.partitionBy("join_date").orderBy(F.desc("weather_readings"))
    dominant_weather = daily_weather.withColumn(
        "rank", F.row_number().over(weather_window)
    ).filter(F.col("rank") == 1).drop("rank", "weather_readings")
    
    # Broadcast join with deliveries
    deliveries_with_weather = deliveries_prepped.join(
        F.broadcast(dominant_weather),
        on="join_date",
        how="left"
    )
    
    # Analyze: delivery count by weather condition
    print("=== Deliveries by Weather Condition ===")
    deliveries_with_weather.groupBy(cond_col).agg(
        F.count("*").alias("delivery_count")
    ).orderBy(F.desc("delivery_count")).show()
    
    # Check the join plan for BroadcastHashJoin
    print("\n=== Join Plan (look for BroadcastHashJoin) ===")
    deliveries_with_weather.explain()
else:
    print("Could not determine join columns — inspect schemas above and adjust.")

### Step 3.7: Write GreenRoute Analytics Output

We will write the curated trip segments and fuel efficiency data as Parquet, then compare file sizes.

In [ ]:
GR_OUTPUT = "/home/jovyan/data/greenroute/curated"

# Write trip segments (partitioned by truck_id for efficient per-truck queries)
trip_metrics.write.mode("overwrite") \
    .partitionBy(TRUCK_COL) \
    .parquet(f"{GR_OUTPUT}/trip_segments")

print("Wrote trip_segments as Parquet (partitioned by truck)")

# Write fuel efficiency summary
if 'truck_efficiency' in dir():
    truck_efficiency.write.mode("overwrite") \
        .parquet(f"{GR_OUTPUT}/fuel_efficiency")
    print("Wrote fuel_efficiency as Parquet")

# Show output directory structure
print("\n=== GreenRoute Curated Output ===")
for root, dirs, files in os.walk(GR_OUTPUT):
    level = root.replace(GR_OUTPUT, "").count(os.sep)
    indent = "  " * level
    print(f"{indent}{os.path.basename(root)}/")
    if level <= 1:
        for f in sorted(files)[:5]:
            print(f"{indent}  {f}")
        if len(files) > 5:
            print(f"{indent}  ... and {len(files) - 5} more files")

In [ ]:
# Compare file sizes: Avro input vs Parquet output
gps_avro_size = sum(
    os.path.getsize(f) for f in glob.glob(
        "/home/jovyan/data/greenroute/gps-telemetry/day_*/hour_*/gps_data.avro"
    )
)
trip_parquet_size = get_dir_size(f"{GR_OUTPUT}/trip_segments") if os.path.exists(f"{GR_OUTPUT}/trip_segments") else 0

print(f"=== GreenRoute Size Comparison ===")
print(f"  GPS Avro input (3 days):    {gps_avro_size / 1024 / 1024:.2f} MB")
print(f"  Trip Parquet output:        {trip_parquet_size / 1024 / 1024:.2f} MB")
if gps_avro_size > 0 and trip_parquet_size > 0:
    print(f"  Compression ratio:          {gps_avro_size / trip_parquet_size:.1f}x")
print("\n(Note: the trip output is a summary -- fewer rows than raw GPS -- so the size")
print("reduction reflects both aggregation and Parquet's columnar compression.)")
print("\n(Also note: the input is already Avro, which is itself ~5x smaller than the")
print("equivalent JSON would be -- so the JSON-equivalent input would be ~5x larger.)")

---

## Part 4: Optimization Exploration

Spark provides several mechanisms to optimize query performance. In this section, we explore:

1. **Caching** — keep frequently-used DataFrames in memory
2. **Query plans** — understand what Spark is actually doing
3. **Partition pruning** — skip irrelevant data when reading Parquet
4. **Repartitioning** — control the number of tasks

### 4a. Caching

When you re-use a DataFrame multiple times, caching avoids re-reading and re-computing it from scratch. Let's time a query before and after caching.

In [ ]:
%%time
# Query WITHOUT caching — Spark reads from disk each time
result1 = all_patients.filter(F.col("department") == "Cardiology").count()
print(f"Cardiology patients (uncached): {result1}")

In [ ]:
# Cache the DataFrame in memory
all_patients.cache()

# The first action after .cache() triggers the caching
all_patients.count()  # This materializes the cache
print("DataFrame cached. Check http://localhost:4040/storage to see it in the Spark UI.")

In [ ]:
%%time
# Query WITH caching — Spark reads from memory
result2 = all_patients.filter(F.col("department") == "Cardiology").count()
print(f"Cardiology patients (cached): {result2}")

In [ ]:
# Clean up cache when done
all_patients.unpersist()
print("Cache released.")

### 4b. Understanding Query Plans

Use `.explain(True)` to see all stages of the Catalyst optimizer: parsed, analyzed, optimized, and physical plans.

In [ ]:
# A moderately complex query to inspect
complex_query = (
    all_patients
    .filter(F.col("age") > 50)
    .join(claims, on="patient_id", how="inner")
    .groupBy("hospital_id", "department")
    .agg(
        F.count("*").alias("claim_count"),
        F.round(F.sum("amount"), 2).alias("total_amount")
    )
    .filter(F.col("claim_count") > 10)
    .orderBy(F.desc("total_amount"))
)

# Show the full plan
print("=== Full Catalyst Plan ===")
complex_query.explain(True)

### 4c. Partition Pruning with Parquet

When Parquet data is partitioned (e.g., by `hospital_id`), Spark can skip entire directories that don't match your filter. This is called **partition pruning**, and it can dramatically reduce I/O.

In [ ]:
# Read the partitioned Parquet we wrote earlier — with a filter
parquet_path = "/home/jovyan/data/healthpulse/curated/unified_patients"

# Without filter — reads all partitions
all_from_parquet = spark.read.parquet(parquet_path)
print(f"All patients from Parquet: {all_from_parquet.count()}")

# With filter on partition column — Spark prunes unneeded partitions
h01_only = spark.read.parquet(parquet_path).filter(F.col("hospital_id") == "hospital_01")

print(f"\nHospital 01 only: {h01_only.count()}")

# Check the plan for PartitionFilters (partition pruning evidence)
print("\n=== Query Plan (look for 'PartitionFilters' and 'PushedFilters') ===")
h01_only.explain()

### 4d. Repartitioning

The number of partitions controls parallelism. Too few partitions underutilize the cluster; too many create overhead. Use `.repartition()` to control this.

In [ ]:
# Check current partitions
print(f"GPS data partitions: {gps_data.rdd.getNumPartitions()}")

# Repartition to a different number
gps_repartitioned = gps_data.repartition(4)
print(f"After repartition(4): {gps_repartitioned.rdd.getNumPartitions()}")

# Coalesce (reduce partitions without full shuffle)
gps_coalesced = gps_data.coalesce(2)
print(f"After coalesce(2): {gps_coalesced.rdd.getNumPartitions()}")

print("\nTip: repartition() causes a full shuffle (expensive but evenly distributes data).")
print("     coalesce() avoids a shuffle (cheaper but may produce uneven partitions).")
print("\nCheck the Spark UI at http://localhost:4040/jobs to see the task count differ.")

---

## Part 5: Reflection Questions

Think through the following questions. These connect the hands-on Spark work you did in this notebook to broader data engineering concepts from the lecture.

---

**Question 1:** Compare reading CSV vs JSON vs Parquet in Spark. Which was fastest and why? Consider schema inference, parsing overhead, and columnar vs row-oriented storage in your answer.

---

**Question 2:** The HealthPulse schema harmonization required custom code to rename columns across four hospital files. How would **Avro's schema evolution** feature (discussed in Week 8) have avoided this problem? What would the pipeline look like differently if all hospitals published in Avro with a shared schema registry?

---

**Question 3:** In the GreenRoute weather join, we used a **broadcast join** because the weather table was small (~3,360 records). What would happen if the weather table were 100 GB instead? How would Spark handle the join differently, and what performance implications would that have?

---

**Question 4:** Looking at the CSV-to-Parquet and JSON-to-Parquet size comparisons from this notebook, project the storage savings if GreenRoute converted **all 3 TB/day** of landing zone data to Parquet. What would the daily Parquet storage requirement be, and what would the annual savings look like at typical cloud storage prices (~$0.023/GB/month for S3 Standard)?

---

## Summary

In this companion notebook, you explored Apache Spark's distributed transformation capabilities through two production-style business scenarios:

| | HealthPulse | GreenRoute |
|---|---|---|
| **Industry** | Healthcare | Logistics |
| **Data sources** | Patient records (4 CSVs), device readings (JSON), insurance claims (CSV), lab results (JSON) | GPS telemetry (partitioned JSON), delivery receipts (nested JSON), weather (JSON), fuel logs (CSV) |
| **Key challenge** | Schema variation across hospitals | Time-series segmentation and multi-source correlation |
| **Transform techniques** | Schema harmonization, joins, anomaly flagging, aggregation | Window functions, explode, broadcast joins, fuel efficiency computation |
| **Output format** | Parquet (partitioned by hospital_id) | Parquet (partitioned by truck_id) |
| **Optimization demonstrated** | Caching, partition pruning | Broadcast join, repartitioning |

**Key takeaways:**

1. **Spark transformations are lazy** — nothing runs until you call an action. This lets the Catalyst optimizer rewrite your query for efficiency.
2. **Schema harmonization** is one of the most common real-world transform tasks. Standardizing column names and types before union is essential.
3. **Parquet output** is dramatically smaller than CSV/JSON input, and supports partition pruning and predicate pushdown for fast downstream queries.
4. **Broadcast joins** eliminate network shuffles when one side is small — always look for this optimization opportunity.
5. **Window functions** are essential for time-series analytics (trip segmentation, running totals, lag/lead comparisons).

**Next week:** We will add the **Stream** layer with Apache Kafka, processing real-time data that feeds into the same Transform pipelines you built today.

In [ ]:
# Clean up: stop the SparkSession
spark.stop()
print("SparkSession stopped. You can now shut down the cluster with:")
print("  docker compose down")